# Fase 3.8 - AIGI pareado, execucao manual no Colab

O piloto, o preparo dos 480 pares, a copia local e o smoke test ja foram concluidos. Para continuar em uma nova sessao: execute as celulas #1 a #5, depois #8 (refaz a copia para /content) e #10 (inicia ou retoma o treino). As celulas #7 e #9 nao precisam ser repetidas. Depois leia #11; com o treino interno concluido, execute #12 a #15 para a comparacao Mirage. Nao use Run All. O treino so inicia quando a celula #10 e executada manualmente.


In [1]:
# 1. GPU e dependências do runtime.
import torch, sys, subprocess
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab antes de continuar'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cryptography', 'huggingface_hub', 'pyarrow', 'scikit-learn'])
print('Dependências disponíveis.', flush=True)

GPU: Tesla T4
Dependências disponíveis.


In [2]:
# 2. Drive e pastas novas; não altera experimentos antigos.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for name in ('credenciais', 'dados_fase3_8', 'modelos', 'resultados'):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT, '| livre reportado:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')

Mounted at /content/drive
Drive: /content/drive/MyDrive/TCC_Final | livre reportado: 5.09 GB
/content livre: 252.95 GB


In [3]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

Credencial desbloqueada nesta sessão.


In [4]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'fase3-ajuste-fino'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase38-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('experimentos/fase3_8_aigi_piloto.py', 'experimentos/fase3_8_aigi_dados.py', 'experimentos/fase3_8_aigi_finetune.py', 'experimentos/avaliar_fase3_8_mirage.py', 'experimentos/fase3_5_dados.py', 'experimentos/fase3_7_dados.py', 'Notebooks/fase_3/fase3.3_compressao.ipynb')
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc37_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro os arquivos da Fase 3.8 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

Baixando snapshot do GitHub...
Código sincronizado: cf83eca970d447a1213c677509443478c7a71f78


In [5]:
# 5. Comandos e progresso periódico; nenhum treino é iniciado aqui.
import os, queue, subprocess, sys, threading, time
DATA = CODE_ROOT / 'experimentos/fase3_8_aigi_dados.py'
TRAIN = CODE_ROOT / 'experimentos/fase3_8_aigi_finetune.py'
for path in (DATA, TRAIN):
    if not path.is_file(): raise FileNotFoundError(path)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command, *, child_env=None):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV if child_env is None else child_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)

Código: /content/tcc_codigo/cf83eca970d447a1213c677509443478c7a71f78


In [7]:
# 6. PILOTO JA EXECUTADO: a saida foi analisada antes deste plano.
print('Piloto AIGI: commit fixado; 16 fotos reais testadas, 15 acessiveis, somente 2 com lado curto >=512. O preparo usa lado curto >=256 e valida cada par.')


In [16]:
# 7. PREPARO MANUAL: plano original + tres shards adicionais, sem perder os pares aceitos.
# Usa a credencial HF cifrada somente para o download, quando disponivel.
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
hf_env = ENV.copy()
hf_path = ROOT / 'credenciais/huggingface.enc'
if hf_path.is_file():
    credential = json.loads(hf_path.read_text(encoding='utf-8'))
    if credential.get('version') != 1 or not {'salt', 'iterations', 'ciphertext'} <= credential.keys():
        raise ValueError('Formato da credencial Hugging Face nao reconhecido')
    if int(credential['iterations']) < 600000:
        raise ValueError('Credencial Hugging Face com KDF insuficiente')
    password = getpass.getpass('Senha de desbloqueio do Hugging Face: ')
    key = base64.urlsafe_b64encode(PBKDF2HMAC(
        algorithm=hashes.SHA256(), length=32,
        salt=base64.b64decode(credential['salt']),
        iterations=int(credential['iterations'])).derive(password.encode()))
    try:
        secret = Fernet(key).decrypt(credential['ciphertext'].encode()).decode()
    except InvalidToken:
        raise ValueError('Senha incorreta ou credencial Hugging Face corrompida') from None
    finally:
        del password, key
    hf_env['HF_TOKEN'] = secret
    del secret
    print('Credencial Hugging Face desbloqueada para esta preparacao.', flush=True)
else:
    print('Credencial Hugging Face cifrada ausente; AIGI publico sera acessado sem autenticacao.', flush=True)
try:
    run_live('preparacao 3.8', [sys.executable, '-u', str(DATA), 'prepare',
                                '--root', str(ROOT)], child_env=hf_env)
finally:
    hf_env.pop('HF_TOKEN', None)


In [6]:
# 8. COPIA LOCAL: execute a cada nova sessao antes do smoke/treino.
OLD_DATA = CODE_ROOT / 'experimentos/fase3_5_dados.py'
PRIOR_DATA = CODE_ROOT / 'experimentos/fase3_7_dados.py'
run_live('copia local 3.5', [sys.executable, '-u', str(OLD_DATA), 'stage', '--root', str(ROOT)])
run_live('copia local 3.7', [sys.executable, '-u', str(PRIOR_DATA), 'stage', '--root', str(ROOT)])
run_live('copia local AIGI 3.8', [sys.executable, '-u', str(DATA), 'stage', '--root', str(ROOT)])


copia local 3.5: iniciando
Cópia/verificação: 100/3000
Cópia/verificação: 200/3000
Cópia/verificação: 300/3000
Cópia/verificação: 400/3000
Cópia/verificação: 500/3000
Cópia/verificação: 600/3000
Cópia/verificação: 700/3000
Cópia/verificação: 800/3000
Cópia/verificação: 900/3000
Cópia/verificação: 1000/3000
Cópia/verificação: 1100/3000
Cópia/verificação: 1200/3000
Cópia/verificação: 1300/3000
Cópia/verificação: 1400/3000
Cópia/verificação: 1500/3000
Cópia/verificação: 1600/3000
Cópia/verificação: 1700/3000
Cópia/verificação: 1800/3000
Cópia/verificação: 1900/3000
Cópia/verificação: 2000/3000
Cópia/verificação: 2100/3000
Cópia/verificação: 2200/3000
Cópia/verificação: 2300/3000
Cópia/verificação: 2400/3000
Cópia/verificação: 2500/3000
Cópia/verificação: 2600/3000
Cópia/verificação: 2700/3000
Cópia/verificação: 2800/3000
Cópia/verificação: 2900/3000
Cópia/verificação: 3000/3000
Dados internos locais prontos: /content/fase3_5_novos_v1
copia local 3.5: concluído em 10.6 min
copia local 3.7:

In [18]:
# 9. SMOKE MANUAL: verifica particoes, hashes, GPU, checkpoint, forward e escrita; nao treina.
run_live('smoke 3.8', [sys.executable, '-u', str(TRAIN), 'smoke'])


In [7]:
# 10. TREINO MANUAL: inicia ou retoma de last.pt, somente apos preparacao do runtime.
required = ('run_live', 'TRAIN', 'ROOT', 'CODE_ROOT', 'ENV', 'sys', 'Path')
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError('Runtime sem configuracao completa (' + ', '.join(missing) + '). Execute #1 a #5 e depois #8 nesta sessao antes de #10.')
local_manifests = [Path('/content/fase3_5_novos_v1/manifest.json'),
                   Path('/content/fase3_7_cfsmall_expansao_v1/manifest.json'),
                   Path('/content/aigi_pares_256_v1/manifest.json')]
if not TRAIN.is_file() or not all(path.is_file() for path in local_manifests):
    raise RuntimeError('Codigo ou dados locais ausentes. Execute #4, #5 e #8 nesta sessao antes de #10.')
run_live('ajuste fino 3.8', [sys.executable, '-u', str(TRAIN), 'auto'])


ajuste fino 3.8: iniciando
GPU: Tesla T4
Arquitetura da Fase 2 montada | ramos: ['vit', 'fft', 'cnn'] | fusão: gate | T2=0.391 | limiar2=0.725 | parâmetros: 10.06M
Partições 3.7: 6152 1420 1420 | teste histórico reservado, não avaliado: 1947
ajuste fino 3.8: ativo há 0.7 min; sem nova saída
ajuste fino 3.8: ativo há 1.2 min; sem nova saída
ajuste fino 3.8: ativo há 1.7 min; sem nova saída
ajuste fino 3.8: ativo há 2.2 min; sem nova saída
Hashes treino: 500/6152
ajuste fino 3.8: ativo há 3.2 min; sem nova saída
ajuste fino 3.8: ativo há 3.7 min; sem nova saída
ajuste fino 3.8: ativo há 4.2 min; sem nova saída
ajuste fino 3.8: ativo há 4.7 min; sem nova saída
Hashes treino: 1000/6152
ajuste fino 3.8: ativo há 5.5 min; sem nova saída
ajuste fino 3.8: ativo há 6.0 min; sem nova saída
ajuste fino 3.8: ativo há 6.5 min; sem nova saída
ajuste fino 3.8: ativo há 7.0 min; sem nova saída
Hashes treino: 1500/6152
ajuste fino 3.8: ativo há 7.9 min; sem nova saída
Hashes treino: 2000/6152
Hashes tr

In [8]:
# 11. LEITURA INTERNA: selecao escolhe epoca; calibracao define T e limiar.
run_dir = ROOT / 'modelos/fase3_8_aigi_pares256_v1'
for name in ('experiment_config.json', 'split_manifest.json', 'history.csv', 'last.pt',
             'best.pt', 'phase3_final.pt', 'internal_metrics.json', 'training_complete.json'):
    print(name, 'presente' if (run_dir / name).is_file() else 'AUSENTE')
metrics = run_dir / 'internal_metrics.json'
print(json.dumps(json.loads(metrics.read_text(encoding='utf-8')), indent=2)
      if metrics.is_file() else 'Treino interno ainda nao concluido.')


experiment_config.json presente
split_manifest.json presente
history.csv presente
last.pt presente
best.pt presente
phase3_final.pt presente
internal_metrics.json presente
training_complete.json presente
{
  "best_epoch": 12,
  "selection_auc": 0.9603094233473981,
  "calibration_bal_acc_apparent": 0.9113622664255576,
  "temperature": 0.5229030847549438,
  "threshold": 0.30999999999999994,
  "train_count": 6152,
  "selection_count": 1420,
  "calibration_count": 1420,
  "external_used": false,
  "openfake_test_used": false
}


## Avaliacao externa comparativa

As celulas 12-15 copiam as mesmas 2.000 imagens Mirage ja congeladas, registram checkpoint, temperatura e limiar antes da inferencia e preservam o resultado da 3.8 em pasta propria. Mirage ja foi observado e nao deve orientar ajustes de treino ou calibracao.


In [9]:
# 12. MIRAGE: copia local verificada, sem novo download da fonte.
MIRAGE_DATA = CODE_ROOT / 'experimentos/benchmark_mirage_2000.py'
run_live('copia local Mirage', [sys.executable, '-u', str(MIRAGE_DATA), 'stage', '--root', str(ROOT)])


copia local Mirage: iniciando
Cópia/verificação: 100/2000
Cópia/verificação: 200/2000
Cópia/verificação: 300/2000
Cópia/verificação: 400/2000
Cópia/verificação: 500/2000
Cópia/verificação: 600/2000
Cópia/verificação: 700/2000
Cópia/verificação: 800/2000
Cópia/verificação: 900/2000
Cópia/verificação: 1000/2000
Cópia/verificação: 1100/2000
Cópia/verificação: 1200/2000
Cópia/verificação: 1300/2000
Cópia/verificação: 1400/2000
Cópia/verificação: 1500/2000
copia local Mirage: ativo há 6.8 min; sem nova saída
Cópia/verificação: 1600/2000
Cópia/verificação: 1700/2000
Cópia/verificação: 1800/2000
Cópia/verificação: 1900/2000
Cópia/verificação: 2000/2000
Benchmark local pronto: /content/benchmark_externo_mirage_2000_v1
copia local Mirage: concluído em 8.5 min


In [10]:
# 13. CONGELAMENTO: checkpoint, temperatura, limiar, codigo e manifesto.
EVAL = CODE_ROOT / 'experimentos/avaliar_fase3_8_mirage.py'
run_live('congelamento 3.8', [sys.executable, '-u', str(EVAL), 'freeze', '--root', str(ROOT)])


congelamento 3.8: iniciando
Configuração 3.8 congelada antes da inferência: /content/drive/MyDrive/TCC_Final/resultados/benchmark_externo_mirage_2000_v1/locks/fase3_8_aigi_pares256_v1.json
{
  "model": "fase3_8_aigi_pares256_v1",
  "checkpoint_sha256": "de4a11cbb733ad9088860d5f254cbb2a900f181cc1ccb28b2fd894cf4a37706a",
  "config_sha256": "fe8ef6e2accfdb9a2871d45b4b1d226d1306a0ba2b7abaf6e0ea6d38e2f8d7c2",
  "internal_metrics_sha256": "2dfb4deb884cadec3beac56afdfef24b0d5cd208eb8ca1395083436acdb69643",
  "architecture_source_sha256": "3db23c93de1130c4b55a1b1ad2f3f522005e5e2dd3d15733698f1f2418eb7c7b",
  "evaluation_script_sha256": "13740967f663e2d953f4fc2ca977d2446497673fe83a939c54b0ca26e4f0adbc",
  "manifest_sha256": "4b3658f880e022957557933fd662c69485c19a5ed9a04e2e09c32ce485737190",
  "prior_metrics_sha256": "0f203fa737c523c3c91944b56d78834c90494851c67aa2a586e82d61c9e3f25e",
  "prior_predictions_sha256": "92d5f18151e3cfa2a19133127f9f0268aa4d028ffdba509adf63aa8ef2ddf296",
  "temperature":

In [11]:
# 14. AVALIACAO EXTERNA COMPARATIVA apos o congelamento.
run_live('avaliacao Mirage 3.8', [sys.executable, '-u', str(EVAL), 'evaluate', '--root', str(ROOT)])


avaliacao Mirage 3.8: iniciando
Integridade antes da inferência: 100/2000
Integridade antes da inferência: 200/2000
Integridade antes da inferência: 300/2000
Integridade antes da inferência: 400/2000
Integridade antes da inferência: 500/2000
Integridade antes da inferência: 600/2000
Integridade antes da inferência: 700/2000
Integridade antes da inferência: 800/2000
Integridade antes da inferência: 900/2000
Integridade antes da inferência: 1000/2000
Integridade antes da inferência: 1100/2000
Integridade antes da inferência: 1200/2000
Integridade antes da inferência: 1300/2000
Integridade antes da inferência: 1400/2000
Integridade antes da inferência: 1500/2000
Integridade antes da inferência: 1600/2000
Integridade antes da inferência: 1700/2000
Integridade antes da inferência: 1800/2000
Integridade antes da inferência: 1900/2000
Integridade antes da inferência: 2000/2000
Fase 3.8 congelada | GPU=Tesla T4 | T=0.522903 | limiar=0.310000
Inferência externa: 100/2000
Inferência externa: 200

In [12]:
# 15. LEITURA do resultado preservado; sem recalibracao.
result = ROOT / 'resultados/benchmark_externo_mirage_2000_v1/fase3_8_aigi_pares256_v1/metrics.json'
print(json.dumps(json.loads(result.read_text(encoding='utf-8')), indent=2)
      if result.is_file() else 'Avaliacao externa ainda nao executada.')


{
  "model": "fase3_8_aigi_pares256_v1",
  "n": 2000,
  "real": 1000,
  "fake": 1000,
  "accuracy": 0.599,
  "accuracy_ci95_wilson": [
    0.5773507421655506,
    0.6202696824727612
  ],
  "balanced_accuracy": 0.599,
  "auc": 0.641268,
  "precision": 0.6080786026200873,
  "recall_fake": 0.557,
  "specificity_real": 0.641,
  "f1": 0.5814196242171189,
  "tn": 641,
  "fp": 359,
  "fn": 443,
  "tp": 557,
  "temperature_frozen": 0.5229030847549438,
  "threshold_frozen": 0.30999999999999994,
  "upscaled_short_side_below_224": 0,
  "checkpoint_sha256": "de4a11cbb733ad9088860d5f254cbb2a900f181cc1ccb28b2fd894cf4a37706a",
  "manifest_sha256": "4b3658f880e022957557933fd662c69485c19a5ed9a04e2e09c32ce485737190",
  "evaluation_script_sha256": "13740967f663e2d953f4fc2ca977d2446497673fe83a939c54b0ca26e4f0adbc",
  "architecture_source_sha256": "3db23c93de1130c4b55a1b1ad2f3f522005e5e2dd3d15733698f1f2418eb7c7b",
  "architecture_extraction_sha256": "ad5db5ee692cd67bc3838ff814b36a179f3eb1acc2cdd77cf5fee41a